# 1. Knowledge Distillation 
· from SAM Large Model encoder to student encoder 

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
from PIL import Image
import cv2
import os
from typing import Dict, List, Tuple, Optional
import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
from types import SimpleNamespace

## 1.1 Data Prepare

In [2]:
data_dic = {
    "CVC_ClinicDB" : 0, # Colonoscopy   - 612 img / 612 mask
    "Darwin": 1,        # X-ray         - 6106 img / 6106 mask
    "DRIVE": 2,         # Fundus        - 40 img / 40 mask
    "DSB_2018": 3,      # Microscopy    - 670 img / 670 mask
    "ISIC_2018": 4,     # Dermoscopy    - 3694 img / 3694 mask
    "Montgomery": 1,    # X-ray         - 138 img / 138 mask
    "PCXA": 1,          # X-ray         - 55 img / 55 mask
    "Shenzhen" : 1,     # X-ray         - 566 img / 566 mask
    "UDIAT" : 5         # Ultrasound    - 163 img / 163 mask
}

In [4]:
def file_generator(folder_path = "../datasets"):
    """
    file path generator
    """
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            yield os.path.join(root, file)

In [5]:
lst = [x for x in file_generator()]
all_images = [x for x in lst if x.split("/")[-2] == "image"]
sorted_image = sorted(all_images, key=lambda x: x.split("/")[-1].split(".")[0])

all_masks = [x for x in lst if x.split("/")[-2] == "mask"]
sorted_mask = sorted(all_masks, key=lambda x: x.split("/")[-1].split(".")[0])

image_mask = list(zip(sorted_image, sorted_mask))

In [7]:
labels = []
for path in sorted_image:
    labels.append(data_dic[path.split("/")[2]])

In [8]:
# from sklearn.model_selection import train_test_split

# X_train, X_val, y_train, y_val = train_test_split(
#     image_mask, labels, 
#     test_size=0.2,
#     train_size=0.8, 
#     random_state=42,
#     shuffle=True,  
#     stratify=labels  
# )


## 1.2 Dataset Class Building

In [8]:
class SAMDistillationDataset(Dataset):
    """SAM知识蒸馏数据集"""
    def __init__(self, 
                 image_mask: list, 
                 image_size: int = 1024,
                 augment: bool = True):
        """
        参数:
        - image_dir: 图片目录
        - mask_dir: 分割掩码目录（可选）
        - image_size: 输入图片大小
        - augment: 是否数据增强
        """
        self.image_mask = image_mask
        self.image_size = image_size
        self.augment = augment
        
        # 获取所有图片
        self.image_files = []
        self.mask_files = []

        for image, mask in image_mask:
            self.image_files.append(image)
            self.mask_files.append(mask)
            
        print(f"数据集加载完成: {len(self.image_files)} 张图片")
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # 加载图片
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        image = np.array(image)
        
        # 加载掩码（如果有）
        mask = None
        if self.mask_files and idx < len(self.mask_files):
            mask_path = self.mask_files[idx]
            mask = Image.open(mask_path).convert('L')
            mask = np.array(mask)
        
        # 数据预处理
        image, mask = self.preprocess(image, mask)
        
        # 转换为tensor
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        image_tensor = self.normalize(image_tensor)
        
        if mask is not None:
            mask_tensor = torch.from_numpy(mask).unsqueeze(0).float() / 255.0
        else:
            mask_tensor = torch.zeros(1, self.image_size, self.image_size)
        
        return {
            'image': image_tensor,
            'mask': mask_tensor,
            'image_path': img_path
        }
    
    def preprocess(self, image: np.ndarray, mask: np.ndarray = None):
        """预处理图片和掩码"""
        # 调整大小
        image = cv2.resize(image, (self.image_size, self.image_size))
        
        if mask is not None:
            mask = cv2.resize(mask, (self.image_size, self.image_size), 
                             interpolation=cv2.INTER_NEAREST)
        
        # 数据增强
        if self.augment and np.random.random() > 0.5:
            # 随机水平翻转
            if np.random.random() > 0.5:
                image = cv2.flip(image, 1)
                if mask is not None:
                    mask = cv2.flip(mask, 1)
            
            # 随机亮度调整
            if np.random.random() > 0.5:
                brightness = np.random.uniform(0.8, 1.2)
                image = np.clip(image * brightness, 0, 255).astype(np.uint8)
            
            # 随机对比度调整
            if np.random.random() > 0.5:
                contrast = np.random.uniform(0.8, 1.2)
                image = np.clip(128 + contrast * (image - 128), 0, 255).astype(np.uint8)
        
        return image, mask
    
    def normalize(self, image: torch.Tensor) -> torch.Tensor:
        """SAM使用的归一化"""
        # SAM使用ImageNet归一化
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (image - mean) / std

# 2. Model Setup

# 2.1 Student model (Small Image Encoder)

In [36]:
class TinyViTEncoder(nn.Module):
    """
    轻量级Vision Transformer作为学生模型
    参考TinyViT架构，但更小
    """
    def __init__(self, 
                 embed_dims=[64, 128, 256, 512],
                 depths=[2, 2, 6, 2],
                 num_heads=[2, 4, 8, 16],
                 window_size=7,
                 drop_path_rate=0.1):
        super().__init__()
        
        # Patch Embedding
        self.patch_embed = nn.Sequential(
            nn.Conv2d(3, embed_dims[0]//2, kernel_size=3, stride=2, padding=1),
            nn.LayerNorm(embed_dims[0]//2),
            nn.GELU(),
            nn.Conv2d(embed_dims[0]//2, embed_dims[0], kernel_size=3, stride=2, padding=1),
            nn.LayerNorm(embed_dims[0])
        )
        
        # Transformer Blocks
        self.stages = nn.ModuleList()
        self.downsample_layers = nn.ModuleList()
        
        for i in range(len(depths)):
            # Downsample
            if i > 0:
                downsample = nn.Sequential(
                    nn.LayerNorm(embed_dims[i-1]),
                    nn.Conv2d(embed_dims[i-1], embed_dims[i], kernel_size=2, stride=2)
                )
                self.downsample_layers.append(downsample)
            else:
                self.downsample_layers.append(nn.Identity())
            
            # Transformer blocks
            stage = nn.ModuleList([
                TransformerBlock(
                    dim=embed_dims[i],
                    num_heads=num_heads[i],
                    window_size=window_size,
                    mlp_ratio=4.0,
                    drop_path=drop_path_rate
                ) for _ in range(depths[i])
            ])
            self.stages.append(stage)
        
        # Neck for feature fusion
        self.neck = nn.Sequential(
            nn.Conv2d(sum(embed_dims), 256, 1),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.LayerNorm(256)
        )
        
        # 输出特征维度
        self.output_dim = 256
    
    def forward(self, x):
        
        # Patch embedding
        x = self.patch_embed(x)
        
        features = []
        for i, (downsample, stage) in enumerate(zip(self.downsample_layers, self.stages)):
            # Downsample
            x = downsample(x)
            
            # Transformer blocks
            for blk in stage:
                x = blk(x)
            
            # 记录多尺度特征
            if i > 0:  # 从第2个stage开始记录
                features.append(x)
        
        # 调整特征图大小并拼接
        target_size = features[-1].shape[2:]
        resized_features = []
        for feat in features:
            if feat.shape[2:] != target_size:
                feat = F.interpolate(feat, size=target_size, mode='bilinear', align_corners=False)
            resized_features.append(feat)
        
        # 拼接多尺度特征
        x = torch.cat(resized_features, dim=1)
        
        # Neck
        x = self.neck(x)
        
        return x


class TransformerBlock(nn.Module):
    """轻量级Transformer Block"""
    def __init__(self, dim, num_heads, window_size, mlp_ratio=4.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, num_heads, window_size)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio))
        
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
    
    def forward(self, x):
        B, C, H, W = x.shape
        x_reshaped = x.flatten(2).transpose(1, 2)  # B, L, C
        
        # Attention
        x_reshaped = x_reshaped + self.drop_path(self.attn(self.norm1(x_reshaped), H, W))
        x_reshaped = x_reshaped + self.drop_path(self.mlp(self.norm2(x_reshaped)))
        
        x = x_reshaped.transpose(1, 2).view(B, C, H, W)
        return x


class WindowAttention(nn.Module):
    """窗口注意力"""
    def __init__(self, dim, num_heads, window_size):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )
        
        # 初始化相对位置偏置
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
    
    def forward(self, x, H, W):
        B, L, C = x.shape
        
        # 划分窗口
        x = x.view(B, H, W, C)
        x_windows = window_partition(x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # 注意力
        B_, N, C = x_windows.shape
        qkv = self.qkv(x_windows).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * (C ** -0.5)
        
        # 添加相对位置偏置
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index(self.window_size)
        ].unsqueeze(0)
        attn = attn + relative_position_bias
        
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        
        # 合并窗口
        x = x.view(-1, self.window_size, self.window_size, C)
        x = window_reverse(x, self.window_size, H, W)
        x = x.view(B, H * W, C)
        
        return x
    
    def relative_position_index(self, window_size):
        coords = torch.stack(torch.meshgrid([
            torch.arange(window_size), 
            torch.arange(window_size)
        ]))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        return relative_position_index


def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class MLP(nn.Module):
    """多层感知机"""
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
    
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class DropPath(nn.Module):
    """随机深度"""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob
    
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        output = x.div(keep_prob) * random_tensor
        return output

## 2.2 Light Mobile Net 

In [10]:
class MobileSAMEncoder(nn.Module):
    """
    基于MobileNetV3的小型Encoder
    更适合移动端部署
    """
    def __init__(self, width_mult=1.0):
        super().__init__()
        
        # 使用预训练的MobileNetV3
        self.backbone = timm.create_model(
            'mobilenetv3_large_100', 
            pretrained=True,
            features_only=True
        )
        
        # 特征通道调整
        in_channels = [24, 40, 80, 160, 960]  # MobileNetV3的特征通道
        out_channels = 256
        
        # 特征融合模块
        self.fusion_blocks = nn.ModuleList()
        for in_channel in in_channels[1:]:  # 使用后4个stage的特征
            block = nn.Sequential(
                nn.Conv2d(in_channel, out_channels, 1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )
            self.fusion_blocks.append(block)
        
        # 特征融合后的卷积
        self.final_conv = nn.Sequential(
            nn.Conv2d(out_channels * 4, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1)
        )
        
        self.output_dim = out_channels
    
    def forward(self, x):
        # 获取多尺度特征
        features = self.backbone(x)
        
        # 调整特征并融合
        fused_features = []
        target_size = features[-1].shape[2:]
        
        for feat, block in zip(features[1:], self.fusion_blocks):  # 跳过第一个低层特征
            # 调整通道数
            feat = block(feat)
            
            # 调整大小
            if feat.shape[2:] != target_size:
                feat = F.interpolate(feat, size=target_size, mode='bilinear', align_corners=False)
            
            fused_features.append(feat)
        
        # 拼接特征
        x = torch.cat(fused_features, dim=1)
        
        # 最终融合
        x = self.final_conv(x)
        
        return x

## 2.3 Knowledge Distillation Loss

In [11]:
class SAMDistillationLoss(nn.Module):
    """SAM知识蒸馏损失函数"""
    def __init__(self, 
                 temperature=4.0,
                 feature_weight=1.0,
                 output_weight=1.0,
                 attention_weight=0.5,
                 contrastive_weight=0.1):
        super().__init__()
        
        self.temperature = temperature
        self.feature_weight = feature_weight
        self.output_weight = output_weight
        self.attention_weight = attention_weight
        self.contrastive_weight = contrastive_weight
        
        # 损失函数
        self.kl_loss = nn.KLDivLoss(reduction='batchmean')
        self.mse_loss = nn.MSELoss()
        self.cos_sim = nn.CosineSimilarity(dim=1)
    
    def forward(self, 
                student_features: torch.Tensor,
                teacher_features: torch.Tensor,
                student_outputs: Optional[torch.Tensor] = None,
                teacher_outputs: Optional[torch.Tensor] = None,
                masks: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        """
        计算蒸馏损失
        
        参数:
        - student_features: 学生模型特征
        - teacher_features: 教师模型特征
        - student_outputs: 学生模型输出（如果有）
        - teacher_outputs: 教师模型输出（如果有）
        - masks: 分割掩码（如果有）
        """
        losses = {}
        
        # 1. 特征蒸馏损失
        if student_features is not None and teacher_features is not None:
            # 特征对齐损失
            feature_loss = self.compute_feature_loss(student_features, teacher_features)
            losses['feature_loss'] = feature_loss * self.feature_weight
        
        # 2. 输出蒸馏损失
        if student_outputs is not None and teacher_outputs is not None:
            # KL散度损失
            output_loss = self.compute_output_loss(student_outputs, teacher_outputs)
            losses['output_loss'] = output_loss * self.output_weight
        
        # 3. 注意力蒸馏损失
        if student_features is not None and teacher_features is not None:
            attention_loss = self.compute_attention_loss(student_features, teacher_features)
            losses['attention_loss'] = attention_loss * self.attention_weight
        
        # 4. 对比学习损失（如果有掩码）
        if masks is not None and student_features is not None:
            contrastive_loss = self.compute_contrastive_loss(student_features, masks)
            losses['contrastive_loss'] = contrastive_loss * self.contrastive_weight
        
        # 总损失
        total_loss = sum(losses.values())
        losses['total_loss'] = total_loss
        
        return losses
    
    def compute_feature_loss(self, student_features, teacher_features):
        """特征蒸馏损失"""
        # 对齐特征维度
        if student_features.shape != teacher_features.shape:
            # 调整学生特征维度
            if student_features.shape[2:] != teacher_features.shape[2:]:
                student_features = F.interpolate(
                    student_features, 
                    size=teacher_features.shape[2:],
                    mode='bilinear',
                    align_corners=False
                )
            
            # 调整通道维度
            if student_features.shape[1] != teacher_features.shape[1]:
                student_features = F.adaptive_avg_pool2d(student_features, 1)
                teacher_features = F.adaptive_avg_pool2d(teacher_features, 1)
        
        # 特征蒸馏损失
        student_norm = F.normalize(student_features, p=2, dim=1)
        teacher_norm = F.normalize(teacher_features, p=2, dim=1)
        
        # 余弦相似度损失
        cos_sim = (student_norm * teacher_norm).sum(dim=1)
        feature_loss = 1 - cos_sim.mean()
        
        return feature_loss
    
    def compute_output_loss(self, student_outputs, teacher_outputs):
        """输出蒸馏损失"""
        # 软化概率分布
        teacher_probs = F.softmax(teacher_outputs / self.temperature, dim=1)
        student_log_probs = F.log_softmax(student_outputs / self.temperature, dim=1)
        
        # KL散度损失
        kl_loss = self.kl_loss(student_log_probs, teacher_probs) * (self.temperature ** 2)
        
        return kl_loss
    
    def compute_attention_loss(self, student_features, teacher_features):
        """注意力蒸馏损失"""
        # 计算空间注意力图
        def get_spatial_attention(features):
            # 对通道维度取L2范数
            attention = torch.norm(features, p=2, dim=1, keepdim=True)
            attention = F.normalize(attention, p=2, dim=[2, 3])
            return attention
        
        student_attention = get_spatial_attention(student_features)
        teacher_attention = get_spatial_attention(teacher_features)
        
        # 注意力图损失
        attention_loss = self.mse_loss(student_attention, teacher_attention)
        
        return attention_loss
    
    def compute_contrastive_loss(self, features, masks):
        """对比学习损失"""
        B, C, H, W = features.shape
        
        # 重塑特征
        features_flat = features.view(B, C, -1).permute(0, 2, 1)  # B, N, C
        masks_flat = F.interpolate(masks, size=(H, W), mode='nearest')
        masks_flat = masks_flat.view(B, 1, -1)  # B, 1, N
        
        # 正负样本对
        loss = 0
        for b in range(B):
            # 获取前景和背景索引
            fg_indices = torch.where(masks_flat[b, 0] > 0.5)[0]
            bg_indices = torch.where(masks_flat[b, 0] <= 0.5)[0]
            
            if len(fg_indices) == 0 or len(bg_indices) == 0:
                continue
            
            # 采样
            n_samples = min(32, len(fg_indices), len(bg_indices))
            fg_idx = fg_indices[torch.randperm(len(fg_indices))[:n_samples]]
            bg_idx = bg_indices[torch.randperm(len(bg_indices))[:n_samples]]
            
            # 特征
            fg_features = features_flat[b, fg_idx]  # n_samples, C
            bg_features = features_flat[b, bg_idx]  # n_samples, C
            
            # 对比损失
            pos_sim = self.cos_sim(fg_features.unsqueeze(1), fg_features.unsqueeze(0))
            neg_sim = self.cos_sim(fg_features.unsqueeze(1), bg_features.unsqueeze(0))
            
            pos_loss = (1 - pos_sim).mean()
            neg_loss = torch.relu(neg_sim - 0.5).mean()  # 希望负样本相似度小于0.5
            
            loss += (pos_loss + neg_loss) / 2
        
        if B > 0:
            loss = loss / B
        
        return loss

# 3. Training Processing

## 3.1 Traininer Class

In [12]:
class SAMDistillationTrainer:
    """SAM知识蒸馏训练器"""
    def __init__(self, 
                 teacher_model,
                 student_model,
                 train_loader,
                 val_loader,
                 device='cuda',
                 lr=1e-4,
                 temperature=4.0,
                 feature_weight=1.0,
                 output_weight=1.0,
                 attention_weight=0.5,
                 contrastive_weight=0.1):
        
        self.device = torch.device(device)
        self.teacher = teacher_model.to(self.device)
        self.student = student_model.to(self.device)
        
        # 冻结教师模型
        for param in self.teacher.parameters():
            param.requires_grad = False
        
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        # 损失函数
        self.criterion = SAMDistillationLoss(
            temperature=temperature,
            feature_weight=feature_weight,
            output_weight=output_weight,
            attention_weight=attention_weight,
            contrastive_weight=contrastive_weight
        )
        
        # 优化器
        self.optimizer = AdamW(
            self.student.parameters(),
            lr=lr,
            weight_decay=1e-4
        )
        
        # 学习率调度器
        self.scheduler = CosineAnnealingLR(
            self.optimizer,
            T_max=len(train_loader) * 50  # 50个epoch
        )
        
        # 记录
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'feature_loss': [],
            'output_loss': [],
            'attention_loss': [],
            'contrastive_loss': []
        }
        
        # 模型保存路径
        self.best_val_loss = float('inf')
    
    def train_epoch(self, epoch):
        """训练一个epoch"""
        self.student.train()
        self.teacher.eval()
        
        total_loss = 0
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}')
        
        for batch_idx, batch in enumerate(pbar):
            # 移动到设备
            images = batch['image'].to(self.device)
            masks = batch['mask'].to(self.device) if 'mask' in batch else None
            
            # 前向传播
            with torch.no_grad():
                teacher_features = self.teacher(images)
            
            student_features = self.student(images)
            
            # 计算损失
            losses = self.criterion(
                student_features=student_features,
                teacher_features=teacher_features,
                masks=masks
            )
            
            loss = losses['total_loss']
            
            # 反向传播
            self.optimizer.zero_grad()
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(self.student.parameters(), max_norm=1.0)
            
            self.optimizer.step()
            self.scheduler.step()
            
            # 记录
            total_loss += loss.item()
            
            # 更新进度条
            pbar.set_postfix({
                'loss': loss.item(),
                'lr': self.optimizer.param_groups[0]['lr']
            })
            
            # 记录各项损失
            for key, value in losses.items():
                if key != 'total_loss':
                    if epoch == 0 and batch_idx == 0:
                        self.history[key] = [value.item()]
                    elif batch_idx < len(self.history[key]):
                        self.history[key][batch_idx] = value.item()
                    else:
                        self.history[key].append(value.item())
        
        avg_loss = total_loss / len(self.train_loader)
        self.history['train_loss'].append(avg_loss)
        
        return avg_loss
    
    def validate(self):
        """验证"""
        self.student.eval()
        self.teacher.eval()
        
        total_loss = 0
        total_feature_loss = 0
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validation'):
                images = batch['image'].to(self.device)
                masks = batch['mask'].to(self.device) if 'mask' in batch else None
                
                teacher_features = self.teacher(images)
                student_features = self.student(images)
                
                losses = self.criterion(
                    student_features=student_features,
                    teacher_features=teacher_features,
                    masks=masks
                )
                
                total_loss += losses['total_loss'].item()
                total_feature_loss += losses.get('feature_loss', 0).item()
        
        avg_loss = total_loss / len(self.val_loader)
        avg_feature_loss = total_feature_loss / len(self.val_loader)
        
        self.history['val_loss'].append(avg_loss)
        
        return avg_loss, avg_feature_loss
    
    def train(self, epochs, save_dir='checkpoints'):
        """训练循环"""
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"开始知识蒸馏训练...")
        print(f"教师模型参数量: {sum(p.numel() for p in self.teacher.parameters()):,}")
        print(f"学生模型参数量: {sum(p.numel() for p in self.student.parameters()):,}")
        print(f"温度参数: {self.criterion.temperature}")
        print(f"特征权重: {self.criterion.feature_weight}")
        print(f"输出权重: {self.criterion.output_weight}")
        
        for epoch in range(epochs):
            # 训练
            train_loss = self.train_epoch(epoch)
            
            # 验证
            val_loss, feature_loss = self.validate()
            
            print(f"Epoch {epoch+1}/{epochs}: "
                  f"Train Loss: {train_loss:.4f}, "
                  f"Val Loss: {val_loss:.4f}, "
                  f"Feature Loss: {feature_loss:.4f}")
            
            # 保存最佳模型
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.save_checkpoint(os.path.join(save_dir, 'best_model.pth'), epoch)
                print(f"保存最佳模型，验证损失: {val_loss:.4f}")
            
            # 定期保存
            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(os.path.join(save_dir, f'checkpoint_epoch{epoch+1}.pth'), epoch)
    
    def save_checkpoint(self, path, epoch):
        """保存检查点"""
        checkpoint = {
            'epoch': epoch,
            'student_state_dict': self.student.state_dict(),
            'teacher_state_dict': self.teacher.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_val_loss': self.best_val_loss,
            'history': self.history
        }
        torch.save(checkpoint, path)
        print(f"模型已保存到: {path}")
    
    def load_checkpoint(self, path):
        """加载检查点"""
        checkpoint = torch.load(path, map_location=self.device)
        self.student.load_state_dict(checkpoint['student_state_dict'])
        self.teacher.load_state_dict(checkpoint['teacher_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.best_val_loss = checkpoint['best_val_loss']
        self.history = checkpoint['history']
        print(f"从 epoch {checkpoint['epoch']} 恢复训练")
    
    def evaluate_similarity(self, data_loader=None):
        """评估学生模型与教师模型的相似度"""
        if data_loader is None:
            data_loader = self.val_loader
        
        self.student.eval()
        self.teacher.eval()
        
        cos_similarities = []
        mse_distances = []
        
        with torch.no_grad():
            for batch in tqdm(data_loader, desc='Evaluating Similarity'):
                images = batch['image'].to(self.device)
                
                teacher_features = self.teacher(images)
                student_features = self.student(images)
                
                # 计算余弦相似度
                teacher_norm = F.normalize(teacher_features, p=2, dim=1)
                student_norm = F.normalize(student_features, p=2, dim=1)
                
                cos_sim = (teacher_norm * student_norm).sum(dim=1).mean().item()
                cos_similarities.append(cos_sim)
                
                # 计算MSE距离
                mse = F.mse_loss(teacher_features, student_features).item()
                mse_distances.append(mse)
        
        avg_cos_sim = np.mean(cos_similarities)
        avg_mse = np.mean(mse_distances)
        
        print(f"特征余弦相似度: {avg_cos_sim:.4f}")
        print(f"特征MSE距离: {avg_mse:.4f}")
        
        return avg_cos_sim, avg_mse

In [13]:
def main(args):
    """主程序"""
    # import argparse
    
    # parser = argparse.ArgumentParser(description='SAM知识蒸馏训练')
    # parser.add_argument('--data_dir', type=str, required=True, help='数据目录')
    # parser.add_argument('--mask_dir', type=str, help='掩码目录（可选）')
    # parser.add_argument('--epochs', type=int, default=50, help='训练轮数')
    # parser.add_argument('--batch_size', type=int, default=8, help='批大小')
    # parser.add_argument('--lr', type=float, default=1e-4, help='学习率')
    # parser.add_argument('--image_size', type=int, default=1024, help='图片大小')
    # parser.add_argument('--temperature', type=float, default=4.0, help='温度参数')
    # parser.add_argument('--student_type', type=str, default='tinyvit', 
    #                    choices=['tinyvit', 'mobile'], help='学生模型类型')
    # parser.add_argument('--device', type=str, default='cuda', help='设备')
    # parser.add_argument('--num_workers', type=int, default=4, help='数据加载线程数')
    # parser.add_argument('--save_dir', type=str, default='checkpoints', help='保存目录')
    
    # args = parser.parse_args()
    
    # 1. 加载数据集
    print("加载数据集...")
    train_dataset = SAMDistillationDataset(
        image_mask=args.data,
        image_size=args.image_size,
        augment=True
    )
    
    # 划分训练集和验证集
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        train_dataset, [train_size, val_size]
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers
    )
    
    print(f"训练集: {len(train_dataset)} 张图片")
    print(f"验证集: {len(val_dataset)} 张图片")
    
    # 2. 加载教师模型（这里需要您提供SAM模型）
    print("\n加载教师模型...")
    
    # 加载 SAM 模型
    teacher_model = load_sam_model('sam_vit_h')
    
    print(f"教师模型参数量: {sum(p.numel() for p in teacher_model.parameters()):,}")
    
    # 3. 初始化学生模型
    print("\n初始化学生模型...")
    if args.student_type == 'tinyvit':
        student_model = TinyViTEncoder(
            embed_dims=[64, 128, 256, 512],
            depths=[2, 2, 6, 2],
            num_heads=[2, 4, 8, 16]
        )
    elif args.student_type == 'mobile':
        student_model = MobileSAMEncoder(width_mult=0.75)
    
    print(f"学生模型参数量: {sum(p.numel() for p in student_model.parameters()):,}")
    print(f"压缩比: {sum(p.numel() for p in teacher_model.parameters()) / sum(p.numel() for p in student_model.parameters()):.2f}x")
    
    # 4. 训练
    print(f"\n开始训练...")
    trainer = SAMDistillationTrainer(
        teacher_model=teacher_model,
        student_model=student_model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=args.device,
        lr=args.lr,
        temperature=args.temperature
    )
    
    trainer.train(epochs=args.epochs, save_dir=args.save_dir)
    
    # 5. 评估
    print(f"\n评估模型相似度...")
    cos_sim, mse_dist = trainer.evaluate_similarity()
    
    # 6. 保存最终模型
    final_path = os.path.join(args.save_dir, 'final_student_model.pth')
    torch.save({
        'model_state_dict': student_model.state_dict(),
        'config': {
            'student_type': args.student_type,
            'image_size': args.image_size,
            'output_dim': student_model.output_dim
        },
        'performance': {
            'cos_similarity': cos_sim,
            'mse_distance': mse_dist,
            'teacher_params': sum(p.numel() for p in teacher_model.parameters()),
            'student_params': sum(p.numel() for p in student_model.parameters())
        }
    }, final_path)
    
    print(f"\n最终模型已保存到: {final_path}")
    print(f"教师模型大小: {sum(p.numel() for p in teacher_model.parameters()):,} 参数")
    print(f"学生模型大小: {sum(p.numel() for p in student_model.parameters()):,} 参数")
    print(f"压缩比例: {sum(p.numel() for p in teacher_model.parameters()) / sum(p.numel() for p in student_model.parameters()):.2f}x")
    print(f"特征相似度: {cos_sim:.4f}")
    
    return student_model


In [14]:
args_set = SimpleNamespace(
    data=image_mask,  # 修改为你的数据路径
    mask_dir=None,  # 如果没有掩码目录，设为None
    epochs=50,
    batch_size=8,
    lr=1e-4,
    image_size=1024,
    temperature=4.0,
    student_type='tinyvit',
    device='cuda',
    num_workers=4,
    save_dir='checkpoints'
)

In [16]:
student_model = main(args_set)

加载数据集...
数据集加载完成: 12044 张图片
训练集: 9635 张图片
验证集: 2409 张图片

加载教师模型...


NameError: name 'load_sam_model' is not defined

In [15]:
def load_sam_model(model_type='vit_h', checkpoint_path=None, device=None):
    """
    加载 SAM 模型
    
    参数:
        model_type: 模型类型，可选 'vit_h', 'vit_l', 'vit_b'
        checkpoint_path: 模型权重路径，如果为None则自动下载
        device: 设备，'cuda' 或 'cpu'，如果为None则自动选择
    
    返回:
        model: SAM 模型
        predictor: SAM 预测器
    """
    
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    
    if model_type not in model_configs:
        raise ValueError(f"model_type 必须是 {list(model_configs.keys())} 之一")
    
    # 如果没有提供 checkpoint 路径，尝试自动下载
    if checkpoint_path is None:
        import os
        checkpoint_path = model_configs[model_type]['default_path']
        if not os.path.exists(checkpoint_path):
            print(f"正在下载 SAM {model_type} 模型...")
            import urllib.request
            urllib.request.urlretrieve(
                model_configs[model_type]['checkpoint_url'],
                checkpoint_path
            )
            print(f"模型已下载到: {checkpoint_path}")
    
    # 加载模型
    print(f"正在加载 SAM {model_type} 模型...")
    sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
    sam.to(device=device)

    return sam

In [16]:
from segment_anything import sam_model_registry, SamPredictor
import matplotlib.pyplot as plt


model_configs = {
        'vit_h': {
            'name': 'vit_h',
            'checkpoint_url': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
            'default_path': '../pretrain/sam_vit_h_4b8939.pth'
        },
        'vit_l': {
            'name': 'vit_l',
            'checkpoint_url': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
            'default_path': '../pretrain/sam_vit_l_0b3195.pth'
        },
        'vit_b': {
            'name': 'vit_b',
            'checkpoint_url': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
            'default_path': 'sam_vit_b_01ec64.pth'
        }
    }

In [17]:
device_ids = list(range(torch.cuda.device_count()))
print(f"可用 GPU: {device_ids}")
model_type = "vit_l"

# 加载模型
teacher_model = sam_model_registry[model_type](checkpoint=model_configs[model_type]["default_path"])

# 1. 先将模型放到主设备
device = torch.device(f"cuda:{device_ids[0]}" if device_ids else "cpu")
teacher_model = teacher_model.to(device)

# 2. 如果多个 GPU，使用 DataParallel
if len(device_ids) > 1:
    teacher_model = nn.DataParallel(teacher_model, device_ids=device_ids)
    print(f"使用 DataParallel 在 {len(device_ids)} 个 GPU 上")

可用 GPU: [0, 1]
使用 DataParallel 在 2 个 GPU 上


In [18]:
teacher_model.eval()

# 4. 关闭梯度
for param in teacher_model.parameters():
    param.requires_grad = False

In [58]:
teacher_model.image_encoder

ImageEncoderViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0-23): 24 x Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): MLPBlock(
        (lin1): Linear(in_features=1024, out_features=4096, bias=True)
        (lin2): Linear(in_features=4096, out_features=1024, bias=True)
        (act): GELU(approximate='none')
      )
    )
  )
  (neck): Sequential(
    (0): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): LayerNorm2d()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): LayerNorm2d()
  )
)

In [19]:
train_dataset = SAMDistillationDataset(
        image_mask=image_mask,
        image_size=1024,
        augment=True
    )

数据集加载完成: 12044 张图片


In [20]:
train_loader = DataLoader(
        train_dataset,
        batch_size=4,
        shuffle=True,
        num_workers=4
    )

In [21]:
for batch in train_loader:
    images = batch['image'].to(device)
    break

In [22]:
images.shape

torch.Size([4, 3, 1024, 1024])

In [24]:
output = teacher_model.module.image_encoder(images)

In [25]:
output.shape

torch.Size([4, 256, 64, 64])

In [ ]:
for batch in train_loader:
    images = batch['image'].to("cuda")
    masks = batch['mask'].to("cuda") if 'mask' in batch else None
    break


import torch
from segment_anything import sam_model_registry
import numpy as np

# 你的输入形状应该是 (B, C, H, W) = (128, 3, 1024, 1024)
batch_size, channels, height, width = images.shape

# 1. 将输入转换为 SAM 期望的格式
# SAM 的 forward 方法期望一个字典列表
batched_input = []
for i in range(batch_size):
    input_dict = {
        'image': images[i],  # 形状: (3, 1024, 1024)
        'original_size': (height, width),  # 原始图像大小
    }
    batched_input.append(input_dict)

# 2. 调用模型
with torch.no_grad():
    outputs = teacher_model(batched_input, multimask_output=True)
    
# 3. 解析输出
# outputs 是一个字典列表，每个字典包含：
# - 'masks': 预测的掩码
# - 'iou_predictions': IoU 预测
# - 'low_res_logits': 低分辨率 logits

In [ ]:
images.shape

In [ ]:
batched_input = []
for i in range(4):
    input_dict = {
        'image': images[i],  # 形状: (3, 1024, 1024)
        'original_size': (1024, 1024),  # 原始图像大小
    }
    batched_input.append(input_dict)

In [143]:
# 检查释放后状态
print(f"释放前 GPU 0 显存: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

释放前 GPU 0 显存: 15.49 GB


In [142]:
# 强制释放所有显存
gc.collect()
torch.cuda.empty_cache()

In [137]:
# 检查释放后状态
print(f"释放后 GPU 0 显存: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

释放后 GPU 0 显存: 15.49 GB


In [37]:
student_encoder = TinyViTEncoder()

In [32]:
student_encoder = nn.DataParallel(student_encoder.to(device), device_ids=device_ids)

In [43]:
student_encoder = student_encoder.to(device)

In [44]:
student_encoder.patch_embed(images)

RuntimeError: Given normalized_shape=[32], expected input with shape [*, 32], but got input of size[4, 32, 512, 512]

In [ ]:
class TinyViTEncoder(nn.Module):
    """
    轻量级Vision Transformer作为学生模型
    参考TinyViT架构，但更小
    """
    def __init__(self, 
                 embed_dims=[64, 128, 256, 512],
                 depths=[2, 2, 6, 2],
                 num_heads=[2, 4, 8, 16],
                 window_size=7,
                 drop_path_rate=0.1):
        super().__init__()
        
        # Patch Embedding
        self.patch_embed = nn.Sequential(
            nn.Conv2d(3, embed_dims[0]//2, kernel_size=3, stride=2, padding=1),
            nn.LayerNorm(embed_dims[0]//2),
            nn.GELU(),
            nn.Conv2d(embed_dims[0]//2, embed_dims[0], kernel_size=3, stride=2, padding=1),
            nn.LayerNorm(embed_dims[0])
        )
        
        # Transformer Blocks
        self.stages = nn.ModuleList()
        self.downsample_layers = nn.ModuleList()
        
        for i in range(len(depths)):
            # Downsample
            if i > 0:
                downsample = nn.Sequential(
                    nn.LayerNorm(embed_dims[i-1]),
                    nn.Conv2d(embed_dims[i-1], embed_dims[i], kernel_size=2, stride=2)
                )
                self.downsample_layers.append(downsample)
            else:
                self.downsample_layers.append(nn.Identity())
            
            # Transformer blocks
            stage = nn.ModuleList([
                TransformerBlock(
                    dim=embed_dims[i],
                    num_heads=num_heads[i],
                    window_size=window_size,
                    mlp_ratio=4.0,
                    drop_path=drop_path_rate
                ) for _ in range(depths[i])
            ])
            self.stages.append(stage)
        
        # Neck for feature fusion
        self.neck = nn.Sequential(
            nn.Conv2d(sum(embed_dims), 256, 1),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.LayerNorm(256)
        )
        
        # 输出特征维度
        self.output_dim = 256
    
    def forward(self, x):
        
        # Patch embedding
        x = self.patch_embed(x)
        
        features = []
        for i, (downsample, stage) in enumerate(zip(self.downsample_layers, self.stages)):
            # Downsample
            x = downsample(x)
            
            # Transformer blocks
            for blk in stage:
                x = blk(x)
            
            # 记录多尺度特征
            if i > 0:  # 从第2个stage开始记录
                features.append(x)
        
        # 调整特征图大小并拼接
        target_size = features[-1].shape[2:]
        resized_features = []
        for feat in features:
            if feat.shape[2:] != target_size:
                feat = F.interpolate(feat, size=target_size, mode='bilinear', align_corners=False)
            resized_features.append(feat)
        
        # 拼接多尺度特征
        x = torch.cat(resized_features, dim=1)
        
        # Neck
        x = self.neck(x)
        
        return x


class TransformerBlock(nn.Module):
    """轻量级Transformer Block"""
    def __init__(self, dim, num_heads, window_size, mlp_ratio=4.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, num_heads, window_size)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio))
        
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
    
    def forward(self, x):
        B, C, H, W = x.shape
        x_reshaped = x.flatten(2).transpose(1, 2)  # B, L, C
        
        # Attention
        x_reshaped = x_reshaped + self.drop_path(self.attn(self.norm1(x_reshaped), H, W))
        x_reshaped = x_reshaped + self.drop_path(self.mlp(self.norm2(x_reshaped)))
        
        x = x_reshaped.transpose(1, 2).view(B, C, H, W)
        return x


class WindowAttention(nn.Module):
    """窗口注意力"""
    def __init__(self, dim, num_heads, window_size):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )
        
        # 初始化相对位置偏置
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
    
    def forward(self, x, H, W):
        B, L, C = x.shape
        
        # 划分窗口
        x = x.view(B, H, W, C)
        x_windows = window_partition(x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # 注意力
        B_, N, C = x_windows.shape
        qkv = self.qkv(x_windows).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * (C ** -0.5)
        
        # 添加相对位置偏置
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index(self.window_size)
        ].unsqueeze(0)
        attn = attn + relative_position_bias
        
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        
        # 合并窗口
        x = x.view(-1, self.window_size, self.window_size, C)
        x = window_reverse(x, self.window_size, H, W)
        x = x.view(B, H * W, C)
        
        return x
    
    def relative_position_index(self, window_size):
        coords = torch.stack(torch.meshgrid([
            torch.arange(window_size), 
            torch.arange(window_size)
        ]))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        return relative_position_index


def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class MLP(nn.Module):
    """多层感知机"""
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
    
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class DropPath(nn.Module):
    """随机深度"""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob
    
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        output = x.div(keep_prob) * random_tensor
        return output